## Cell 1: Setup
**What this demonstrates**: Load API keys, initialise Anthropic/OpenAI/Cohere clients, and configure which reranker to use (Cohere cross-encoder or LLM fallback).
**Expected output**: ✓ Setup complete with LLM model, embedding model, reranker mode, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# cohere: cross-encoder reranker API (optional; LLM fallback used if key absent)
%pip install -q langchain langchain-openai langchain-anthropic langchain-community chromadb python-dotenv tiktoken cohere

# ── Standard library ─────────────────────────────────────────────────────────
import os
import re
import time
import hashlib
import pathlib
from typing import Any

# ── Third-party ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
import cohere as cohere_sdk

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())

_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
_cohere_key    = os.environ.get('COHERE_API_KEY', '')

assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Cohere key is optional — if absent, LLM-based reranking is used as fallback
USE_COHERE = bool(_cohere_key)

# ── Constants ─────────────────────────────────────────────────────────────────
LLM_MODEL   = 'claude-sonnet-4-6'
EMBED_MODEL = 'text-embedding-3-small'
CHROMA_DIR  = './chroma_advanced_rag'

# ── Client initialisation ──────────────────────────────────────────────────
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)
# co_client is None when COHERE_API_KEY is absent
co_client = cohere_sdk.Client(_cohere_key) if USE_COHERE else None

rerank_mode = 'cohere-rerank-english-v3.0' if USE_COHERE else 'llm-fallback'

print('✓ Setup complete')
print(f'  LLM model:       {LLM_MODEL}')
print(f'  Embedding model: {EMBED_MODEL}')
print(f'  Reranker:        {rerank_mode}')
print(f'  Anthropic key:   ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:      ...{_openai_key[-4:]}')
if USE_COHERE:
    print(f'  Cohere key:      ...{_cohere_key[-4:]}')
else:
    print('  Cohere key:      not set — using LLM fallback reranker')

## Cell 2: Data — Basel III Regulatory Excerpt
**What this demonstrates**: Load a structured regulatory document where query vocabulary systematically diverges from document phrasing — the core failure mode Advanced RAG is designed to fix.
**Expected output**: Document name, character/word count, first 200 characters, and explanation of why this document exposes Naive RAG's weakness.

In [ ]:
# ── Locate the Basel III document ──────────────────────────────────────────────
# Try notebook-relative path first, then repo-root-relative
_candidates = [
    pathlib.Path('../../shared/sample_data/basel_iii_excerpt.txt'),
    pathlib.Path('shared/sample_data/basel_iii_excerpt.txt'),
]
SAMPLE_DATA = next((p for p in _candidates if p.exists()), None)
assert SAMPLE_DATA is not None, (
    'Cannot find basel_iii_excerpt.txt. '
    'Run from the repo root or from modules/02_advanced_rag/.'
)

document_text: str = SAMPLE_DATA.read_text(encoding='utf-8')

print(f'Document:  {SAMPLE_DATA.name}')
print(f'Size:      {len(document_text):,} characters  |  ~{len(document_text.split()):,} words')
print()
print('First 200 characters:')
print('-' * 52)
print(document_text[:200])
print('-' * 52)
print()
print('Why this document?')
print('  Basel III uses formal regulatory phrasing that diverges from natural queries.')
print('  An analyst asking "Tier 1 capital requirements" must retrieve text that says')
print('  "minimum CET1 ratio of 4.5% of risk-weighted assets" — a vocabulary gap')
print('  that cosine similarity alone cannot bridge reliably. Query rewriting + reranking')
print('  are the targeted fixes.')

## Cell 3: Core — Advanced RAG Pipeline
**What this demonstrates**: All four Advanced RAG stages: (1) LLM query rewriting into 3 variants, (2) multi-variant retrieval with deduplication, (3) cross-encoder reranking against the original query, (4) extractive context compression.
**Expected output**: Chunk count after indexing and confirmation that all four pipeline functions are defined.

In [ ]:
# ── Build the vector index ───────────────────────────────────────────────────
# Advanced RAG changes nothing at index time — the same Chroma index as Naive RAG
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ''],
    chunk_size=500,
    chunk_overlap=50,
)
chunks: list[str] = splitter.split_text(document_text)
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name='advanced_rag_basel',
    persist_directory=CHROMA_DIR,
    collection_metadata={'hnsw:space': 'cosine'},
)
print(f'Indexed {len(chunks)} chunks into Chroma')
print()


# ── Stage 1: Query rewriting ───────────────────────────────────────────────
def rewrite_query(query: str, n_variants: int = 3) -> list[str]:
    """Generate N alternative phrasings to broaden retrieval recall.

    Instructs the LLM to produce three variants: one using formal regulatory
    terminology, one using synonyms, and one as the most specific sub-question.
    Returns [original] + variants so the original is always included.

    Args:
        query: The user's original question.
        n_variants: Number of alternative phrasings to generate.

    Returns:
        List starting with the original query followed by up to n_variants alternatives.
    """
    prompt = (
        f'Generate {n_variants} alternative phrasings of the following question '
        f'to improve retrieval from a regulatory document. '
        f'Strategy: use formal regulatory terminology in the first, synonyms in the '
        f'second, and the most specific sub-question in the third. '
        f'Return only the {n_variants} variants, one per line, no numbering.'
        f'\n\nQuestion: {query}'
    )
    response = client.messages.create(
        model=LLM_MODEL,
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}],
    )
    variants = [
        v.strip()
        for v in response.content[0].text.strip().splitlines()
        if v.strip()
    ]
    # Original query is always first — reranker scores against it in Stage 3
    return [query] + variants[:n_variants]


# ── Stage 2: Multi-query retrieval with deduplication ───────────────────────
def retrieve_multi_query(queries: list[str], k: int = 10) -> list[Any]:
    """Retrieve candidates for each query variant; deduplicate by content hash.

    Running k=10 per variant produces ~30-40 raw candidates. Deduplication on
    MD5 hash of stripped content ensures the reranker sees each chunk once,
    preventing duplicate chunks from inflating their apparent relevance.

    Args:
        queries: Query strings (original + rewritten variants).
        k: Candidates to retrieve per query string.

    Returns:
        Deduplicated list of LangChain Document objects.
    """
    seen: set[str] = set()
    docs: list[Any] = []
    for q in queries:
        for doc in vectorstore.similarity_search(q, k=k):
            h = hashlib.md5(doc.page_content.strip().encode()).hexdigest()
            if h not in seen:
                seen.add(h)
                docs.append(doc)
    return docs


# ── Stage 3: Cross-encoder reranking ─────────────────────────────────────
def rerank(
    original_query: str, docs: list[Any], top_n: int = 3
) -> list[tuple[Any, float]]:
    """Rerank candidates against the original (unrewritten) query.

    Always scores against the ORIGINAL query — not a rewritten variant. The
    reranker's job is to reflect user intent; using a rewritten query here
    would optimise for the rewriter's interpretation instead.

    Uses Cohere cross-encoder if COHERE_API_KEY is set; falls back to LLM
    relevance scoring (slower, ~90% accuracy vs Cohere on regulatory text).

    Args:
        original_query: The user's unmodified question.
        docs: Deduplicated candidates from retrieve_multi_query().
        top_n: Number of top documents to return.

    Returns:
        List of (Document, relevance_score) sorted by descending score.
    """
    if USE_COHERE and co_client is not None:
        # Cross-encoder jointly encodes query + doc — far more accurate than cosine
        results = co_client.rerank(
            model='rerank-english-v3.0',
            query=original_query,
            documents=[doc.page_content for doc in docs],
            top_n=top_n,
        )
        return [(docs[r.index], r.relevance_score) for r in results.results]
    # LLM fallback: ask Claude to score each chunk's relevance
    scored: list[tuple[Any, float]] = []
    for doc in docs:
        resp = client.messages.create(
            model=LLM_MODEL,
            max_tokens=8,
            system='Rate how well the text answers the question. Reply with only a decimal from 0.0 to 1.0.',
            messages=[{
                'role': 'user',
                'content': f'Question: {original_query}\n\nText: {doc.page_content[:400]}',
            }],
        )
        try:
            score = float(resp.content[0].text.strip())
        except ValueError:
            score = 0.0
        scored.append((doc, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


# ── Stage 4: Context compression ─────────────────────────────────────────
def compress_chunk(query: str, chunk_text: str) -> str:
    """Extract only the sentences that directly answer the query.

    Mimics LangChain's LLMChainExtractor behaviour: the LLM returns only
    query-relevant sentences verbatim. Explicitly instructs the model to
    preserve conditional language ('unless', 'except', 'subject to') that
    carries critical regulatory meaning and must not be silently dropped.

    Args:
        query: The user's original question.
        chunk_text: Raw text of a top-ranked document chunk.

    Returns:
        Compressed string with only relevant sentences, or the original
        chunk if the model returns fewer than 30 characters (failure guard).
    """
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=300,
        system=(
            'Extract only the sentences from the text that directly answer the question. '
            'Return them verbatim — do not paraphrase or summarise. '
            'Preserve all conditional language: unless, except, subject to, provided that. '
            'If no sentences are directly relevant, return the text unchanged.'
        ),
        messages=[{
            'role': 'user',
            'content': f'Question: {query}\n\nText:\n{chunk_text}',
        }],
    )
    compressed = resp.content[0].text.strip()
    # Fall back to original if compression returns something too short to be useful
    return compressed if len(compressed) > 30 else chunk_text


print('Pipeline functions defined:')
print(f'  Stage 1 ─ rewrite_query          (LLM → 3 variants)')
print(f'  Stage 2 ─ retrieve_multi_query   (k=10 per variant, dedup)')
print(f'  Stage 3 ─ rerank                 ({rerank_mode})')
print(f'  Stage 4 ─ compress_chunk         (LLM extractive compression)')

## Cell 4: Run — End-to-End Execution
**What this demonstrates**: The full Advanced RAG pipeline from query to answer, with all four stages executed sequentially and per-stage timing captured.
**Expected output**: Query variants, candidate count, reranked top-3 chunk previews, the generated answer, and a timing breakdown across all five pipeline stages.

In [ ]:
QUERY = 'What capital requirements apply to tier 1 banks?'

# ── Stage 1: Rewrite the query ───────────────────────────────────────────────
t0 = time.perf_counter()
all_queries: list[str] = rewrite_query(QUERY)
t_rewrite = time.perf_counter()

# ── Stage 2: Retrieve candidates for all variants ─────────────────────────
candidates: list[Any] = retrieve_multi_query(all_queries, k=10)
t_retrieve = time.perf_counter()

# ── Stage 3: Rerank against the ORIGINAL query ───────────────────────────
# Pass QUERY (not all_queries) — reranker scores against user intent, not rewriter output
reranked: list[tuple[Any, float]] = rerank(QUERY, candidates, top_n=3)
t_rerank = time.perf_counter()

# ── Stage 4: Compress each top-ranked chunk ──────────────────────────────
compressed_chunks: list[tuple[str, float]] = [
    (compress_chunk(QUERY, doc.page_content), score)
    for doc, score in reranked
]
t_compress = time.perf_counter()

# ── Assemble prompt from compressed context and generate ────────────────────
context = '\n\n---\n\n'.join(
    f'[Source {i+1} | relevance={score:.3f}]\n{text}'
    for i, (text, score) in enumerate(compressed_chunks)
)
SYSTEM = (
    'You are a regulatory compliance assistant. '
    'Answer using ONLY the provided context. '
    'Cite specific articles and ratio thresholds. '
    'If the context is insufficient, say so explicitly.'
)
response = client.messages.create(
    model=LLM_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {QUERY}'}],
)
answer: str = response.content[0].text
t_end = time.perf_counter()

# ── Print results ────────────────────────────────────────────────────────────────
print(f'Query: {QUERY!r}')
print()
print(f'Query variants ({len(all_queries)} total including original):')
for i, q in enumerate(all_queries):
    label = 'original' if i == 0 else f'variant {i}'
    print(f'  [{label}]  {q}')
print()
print(f'Candidates retrieved: {len(candidates)} unique chunks (deduped across all variants)')
print(f'After reranking:      top {len(reranked)} chunks')
print()
print('Top-ranked chunks (compressed):')
for i, (text, score) in enumerate(compressed_chunks, 1):
    preview = text[:100].replace('\n', ' ')
    print(f'  [{i}] score={score:.3f}  {preview!r}...')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(answer)
print('=' * 65)
print()
ms = lambda a, b: f'{(b - a) * 1000:.0f} ms'
print(
    f'Timing:  rewrite={ms(t0, t_rewrite)}  '
    f'retrieve={ms(t_rewrite, t_retrieve)}  '
    f'rerank={ms(t_retrieve, t_rerank)}  '
    f'compress={ms(t_rerank, t_compress)}  '
    f'generate={ms(t_compress, t_end)}  '
    f'total={ms(t0, t_end)}'
)

## Cell 5: Inspect — Reranking, Compression, and Timing
**What this demonstrates**: How reranking changes document order vs. naive cosine retrieval, how much compression reduces token count, and where time is spent across the five pipeline stages.
**Expected output**: Before/after reranking comparison table, per-chunk compression ratios, token budget comparison (naive vs advanced), and a timing breakdown bar chart.

In [ ]:
# ── Naive vs Advanced: retrieval order comparison ──────────────────────────
# Run naive retrieval now (original query only, cosine similarity, no reranking)
naive_raw = vectorstore.similarity_search_with_score(QUERY, k=5)
# similarity_search_with_score returns (doc, cosine_distance); convert to similarity
naive_results: list[tuple[Any, float]] = [(doc, 1.0 - dist) for doc, dist in naive_raw]

print('Retrieval order: naive cosine (left) vs cross-encoder reranked (right)')
print('-' * 65)
print(f'  {"NAIVE TOP-5 (cosine)": <32}  {"ADVANCED TOP-3 (reranked)"}')
print('-' * 65)
max_rows = max(len(naive_results), len(reranked))
for i in range(max_rows):
    if i < len(naive_results):
        nd, ns = naive_results[i]
        naive_col = f'[{i+1}] {ns:.3f}  {nd.page_content[:28].replace(chr(10), " ")}...'
    else:
        naive_col = ''
    if i < len(reranked):
        rd, rs = reranked[i]
        rank_col = f'[{i+1}] {rs:.3f}  {rd.page_content[:28].replace(chr(10), " ")}...'
    else:
        rank_col = ''
    print(f'  {naive_col: <32}  {rank_col}')

# ── Compression ratio per chunk ───────────────────────────────────────
print()
print('Context compression: chars before \u2192 after')
print('-' * 65)
total_before = total_after = 0
for i, ((compressed_text, score), (orig_doc, _)) in enumerate(
    zip(compressed_chunks, reranked), 1
):
    before = len(orig_doc.page_content)
    after  = len(compressed_text)
    pct    = (1 - after / before) * 100 if before else 0
    total_before += before
    total_after  += after
    bar = '\u2588' * int(pct / 5)  # each block = 5% reduction
    print(f'  [Chunk {i}]  {before:>4} \u2192 {after:>4} chars  ({pct:.0f}% reduced)  {bar}')
overall_pct = (1 - total_after / total_before) * 100 if total_before else 0
print(f'  {"Overall":<10}  {total_before:>4} \u2192 {total_after:>4} chars  ({overall_pct:.0f}% reduced)')

# ── Token budget: naive vs advanced ───────────────────────────────────
enc = tiktoken.get_encoding('cl100k_base')
count_tok = lambda t: len(enc.encode(t))

naive_ctx_tok = count_tok('\n\n'.join(doc.page_content for doc, _ in naive_results))
adv_ctx_tok   = count_tok('\n\n'.join(text for text, _ in compressed_chunks))
ans_tok       = count_tok(answer)

print()
print('Token budget comparison:')
print(f'  {"":30}  {"Naive RAG":>10}  {"Advanced RAG":>12}')
print(f'  {"Chunks in context":30}  {"5":>10}  {str(len(compressed_chunks)):>12}')
print(f'  {"Context tokens":30}  {naive_ctx_tok:>10}  {adv_ctx_tok:>12}')
print(f'  {"Context token reduction":30}  {"---":>10}  {"-" + str(naive_ctx_tok - adv_ctx_tok) + " tok":>12}')

# ── Timing breakdown ───────────────────────────────────────────────────
timings = {
    'rewrite':  (t_rewrite  - t0)          * 1000,
    'retrieve': (t_retrieve - t_rewrite)   * 1000,
    'rerank':   (t_rerank   - t_retrieve)  * 1000,
    'compress': (t_compress - t_rerank)    * 1000,
    'generate': (t_end      - t_compress)  * 1000,
}
total_ms = sum(timings.values())
print()
print(f'Timing breakdown (total = {total_ms:.0f} ms):')
for stage, ms_val in timings.items():
    pct = ms_val / total_ms if total_ms else 0
    bar = '\u2588' * int(pct * 40)
    print(f'  {stage:10}  {ms_val:>6.0f} ms  {bar}')

## Cell 6: Fintech Application — Regulatory Compliance Q&A with Citation Tracking
**What this demonstrates**: Advanced RAG applied to a G-SIB surcharge compliance question, with citation tracking that verifies each Article reference in the answer against the retrieved source chunks — a basic hallucination guard.
**Expected output**: Query variants, top sources, compliance answer with Article citations, citation grounding audit, and pipeline timing.

In [ ]:
COMPLIANCE_QUERY = (
    'What additional capital surcharges apply to systemically important banks '
    'and how are they determined?'
)

COMPLIANCE_SYSTEM = (
    'You are a Basel III compliance specialist. '
    'Answer using only the provided regulatory excerpts. '
    'For every factual claim, cite the Article number in [Article X.Y] format. '
    'State all numeric thresholds exactly as they appear in the source.'
)


def run_advanced_pipeline(
    query: str,
    system_prompt: str,
    retrieve_k: int = 10,
    top_n: int = 3,
) -> dict[str, Any]:
    """Execute all four Advanced RAG stages and return intermediate outputs.

    Args:
        query: The compliance question.
        system_prompt: System instruction for the generation step.
        retrieve_k: Candidates per query variant.
        top_n: Chunks to pass to generation after reranking.

    Returns:
        Dict with keys: queries, candidates, reranked, compressed, answer, timing.

    Fintech note:
        In production, log every query/answer/citation triple for audit trail.
        Compliance teams need to reproduce any answer from its source excerpts.
    """
    t_start = time.perf_counter()
    queries = rewrite_query(query)
    t1 = time.perf_counter()

    cands = retrieve_multi_query(queries, k=retrieve_k)
    t2 = time.perf_counter()

    top_docs = rerank(query, cands, top_n=top_n)
    t3 = time.perf_counter()

    compressed = [
        (compress_chunk(query, doc.page_content), score)
        for doc, score in top_docs
    ]
    t4 = time.perf_counter()

    context = '\n\n---\n\n'.join(
        f'[Excerpt {i+1} | relevance={score:.3f}]\n{text}'
        for i, (text, score) in enumerate(compressed)
    )
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=600,
        system=system_prompt,
        messages=[{
            'role': 'user',
            'content': f'Regulatory excerpts:\n{context}\n\nCompliance question: {query}',
        }],
    )
    t5 = time.perf_counter()
    ms_int = lambda a, b: round((b - a) * 1000)

    return {
        'query':      query,
        'queries':    queries,
        'candidates': cands,
        'reranked':   top_docs,
        'compressed': compressed,
        'answer':     resp.content[0].text,
        'timing': {
            'rewrite_ms':  ms_int(t_start, t1),
            'retrieve_ms': ms_int(t1, t2),
            'rerank_ms':   ms_int(t2, t3),
            'compress_ms': ms_int(t3, t4),
            'generate_ms': ms_int(t4, t5),
            'total_ms':    ms_int(t_start, t5),
        },
    }


# ── Execute ─────────────────────────────────────────────────────────────────
result = run_advanced_pipeline(COMPLIANCE_QUERY, COMPLIANCE_SYSTEM)

print('=' * 65)
print('REGULATORY COMPLIANCE Q&A — G-SIB Surcharge Analysis')
print('=' * 65)
print(f'\nQuestion: {result["query"]}')
print(f'\nQuery variants ({len(result["queries"])} total \u2192 {len(result["candidates"])} unique candidates):')
for i, q in enumerate(result['queries']):
    label = 'original' if i == 0 else f'variant {i}'
    print(f'  [{label}]  {q}')

print(f'\nTop {len(result["reranked"])} sources after reranking:')
for i, (doc, score) in enumerate(result['reranked'], 1):
    # Use the first non-blank line as a section label
    label = next((ln.strip() for ln in doc.page_content.split('\n') if ln.strip()), '')
    print(f'  [{i}] score={score:.3f}  \'{label[:58]}\'')

print('\nCompliance answer (with Article citations):')
print('-' * 65)
print(result['answer'])
print('-' * 65)

# ── Citation grounding audit ──────────────────────────────────────────
# Extract all "Article X.Y" citations from the generated answer
cited = re.findall(r'Article\s+\d+\.\d+', result['answer'])
print()
print('Citation grounding audit (hallucination check):')
if not cited:
    print('  No Article citations found in answer.')
else:
    for citation in cited:
        # Check that the cited Article number appears in at least one source chunk
        # Uses the original (uncompressed) chunks from reranked for broader coverage
        found_in_source = any(
            citation in doc.page_content
            for doc, _ in result['reranked']
        )
        status = '\u2713 grounded in retrieved source' if found_in_source else '\u26a0 NOT found in retrieved context'
        print(f'  {citation}: {status}')

t = result['timing']
print()
print(
    f'Timing: {t["rewrite_ms"]} ms rewrite | {t["retrieve_ms"]} ms retrieve | '
    f'{t["rerank_ms"]} ms rerank | {t["compress_ms"]} ms compress | '
    f'{t["generate_ms"]} ms generate | {t["total_ms"]} ms total'
)